In [13]:
import os
import json
import openai
from tqdm import tqdm

# Load your articles JSON (with abstracts included)
with open('articles_with_abstracts.json') as f:
    articles = json.load(f)
    print(f"Loaded {len(articles)} articles.")

with open('tagged_articles_llm.json', 'r') as f:
    tagged_articles = json.load(f)
    print(f"Loaded {len(tagged_articles['articles'])} tagged articles.")

count = 0
for lh in tagged_articles['articles']:
    for rh in articles:
        if lh["title"] == rh["title"] and lh["author"] == rh["author"]:
            rh["tags"] = lh["tags"]
            rh["weightedSum"] = lh["weightedSum"]
            count += 1
            break
    else:
        articles.append(lh)

print(f"Matched {count} articles with tags.")

# with open('refined_tags.json', 'r') as f:
#     tags = json.load(f)
tags = tagged_articles['tags']

# Prepare OpenAI parameters
openai.api_key = os.getenv("OPENAI_API_KEY")
MODEL = "gpt-4o-mini"

def get_relevance_score(tag_desc, abstract_text):
    prompt = (
        f"Rate the relevance of the following article abstract to the tag description on a scale of 0-10.\n\n"
        f"Tag description:\n{tag_desc}\n\n"
        f"Article abstract:\n{abstract_text}\n\n"
        "Respond with only a single integer between 0 (not relevant) and 10 (highly relevant)."
    )
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return int(response.choices[0].message.content.strip())

# Assign secondary weights via LLM
for article in tqdm(articles, desc="Processing articles"):
    if "weightedSum" in article:
        continue
    article["tags"] = []
    weighted_sum = 0
    for tag in tags:
        score = get_relevance_score(tag["description"], article["abstract"])
        if score > 0:
            article["tags"].append({
                "id": tag["id"],
                "secondaryWeight": score
            })
            weighted_sum += tag["primaryWeight"] + score
    article["weightedSum"] = weighted_sum



Loaded 202 articles.
Loaded 82 tagged articles.
Matched 82 articles with tags.


Processing articles: 100%|██████████| 202/202 [30:13<00:00,  8.98s/it]


In [ ]:
# Sort by total weight
articles_sorted = sorted(
    [a for a in articles if "weightedSum" in a], 
    key=lambda x: x["weightedSum"], reverse=True)

# Output final JSON
output = {
    "tags": tags,
    "articles": articles_sorted
}
with open('tagged_articles_llm.json', 'w') as f:
    json.dump(output, f, indent=2)

print("LLM-scored tagged articles JSON generated at tagged_articles_llm.json")
print(f"Total articles written: {len(articles_sorted)}")

LLM-scored tagged articles JSON generated at tagged_articles_llm_2.json
Total articles written: 202


In [22]:
def filter_by_relevance(articles, threshold=0, n=4):
    return [
        article for article in articles 
        if len([tag for tag in article["tags"] if tag["secondaryWeight"] >= threshold]) >= n
    ]

filtered_articles = articles_sorted[:20]
for i in range(1, 11):
    test = filter_by_relevance(articles_sorted[20:], i, 6)
    print(f"Filtered articles with threshold {i}: {len(test)}")
    if len(test) < 25:
        filtered_articles += test
        break

print(f"Filtered articles: {len(filtered_articles)}")
with open('filtered_articles.json', 'w') as f:
    # remove abstracts and tags from filtered articles
    filtered_output = sorted([
        {k: v for k, v in article.items() if k not in ["abstract", "tags"]} for article in filtered_articles
    ], key=lambda x: x["author"])
    json.dump(filtered_output, f, indent=2)

Filtered articles with threshold 1: 180
Filtered articles with threshold 2: 180
Filtered articles with threshold 3: 142
Filtered articles with threshold 4: 92
Filtered articles with threshold 5: 53
Filtered articles with threshold 6: 37
Filtered articles with threshold 7: 18
Filtered articles: 38
